# Week 03 — BBO capstone driver

Round 3. Two points per function. The question this round asks is **local sensitivity**: if I move a small distance from the W2 point, does the output move at all?

- **F1, F2, F5, F6, F8** — perturbation of order 10^-3 around the W2 point. A pair of near-identical queries gives a finite difference, which is a signed direction for the price of one query.
- **F7** — a larger move on x5 and x6 only, isolating those two axes.
- **F3, F4** — abandon the W2 point entirely; both returned poorly, so a fresh region is worth more than local structure around a bad value.

**The flaw, stated now because it recurs:** a 10^-3 perturbation on a surface whose length-scale is order 0.1 returns the same value to four significant figures. That is not a finite difference, it is a repeated measurement — and I spend several later rounds learning this the expensive way.

In [ ]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
# Walk up until bbo.py is found, so the notebook runs from anywhere in the repo.
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "bbo.py")) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root); sys.path.insert(0, _root)
import numpy as np
import pandas as pd
import bbo

WEEK = 3
PRIOR = WEEK - 1          # data state this round was proposed from
SEED = 3
OUTDIR = f"outputs/week{WEEK:02d}"; os.makedirs(OUTDIR, exist_ok=True)

# What each function is getting this round, and why.
PLAN = {
    1: '1e-3 perturbation of W2',
    2: '1e-3 perturbation of W2',
    3: 'relocate — W2 region was poor',
    4: 'relocate — W2 region was poor',
    5: 'small perturbation of W2',
    6: '1e-3 perturbation of W2',
    7: 'isolate x5, x6',
    8: '1e-4 perturbation of W2',
}
pd.DataFrame([dict(func=f"F{f}", d=bbo.DIMS[f], move=PLAN[f]) for f in bbo.FUNC_IDS])


## 1. Data — the state this round was proposed from

Best point on record per function, truncated to rounds ≤ 2. Nothing below this cell may look at later rounds.


In [ ]:
# The ledger comes FIRST every round: the best point on record, not the latest one.
led = bbo.ledger(up_to=PRIOR)
led["best"] = led["best"].map(lambda v: f"{v:.6g}")
led["x"] = led["x"].map(bbo.submission)
led


## 2. Proposals — perturb where W2 was informative, relocate where it was not

In [ ]:
proposals = {
    1: np.array([0.018917, 0.257418]),
    2: np.array([0.098659, 0.954717]),
    3: np.array([0.008917, 0.791931, 0.253392]),
    4: np.array([0.313981, 0.427787, 0.451776, 0.356097]),
    5: np.array([0.013742, 0.287641, 0.718987, 0.211753]),
    6: np.array([0.398757, 0.385494, 0.571114, 0.711617, 0.389851]),
    7: np.array([0.151868, 0.148658, 0.072557, 0.257384, 0.201157, 0.787141]),
    8: np.array([0.159174, 0.118198, 0.137941, 0.716499, 0.782017, 0.543671, 0.279601, 0.258415]),
}

prev = {fid: np.array(bbo.HISTORY[2][fid][0]) for fid in bbo.FUNC_IDS}
pd.DataFrame([dict(func=f"F{fid}",
                   step=float(np.linalg.norm(proposals[fid] - prev[fid])),
                   kind="perturbation" if np.linalg.norm(proposals[fid]-prev[fid]) < 0.05
                        else "relocation",
                   submission=bbo.submission(proposals[fid]))
              for fid in bbo.FUNC_IDS])


### Surrogate trust check

Run before reading any acquisition value, not after.


In [ ]:
# Is each surrogate worth listening to? LOO R2 < 0 means it is worse than
# predicting the mean, and any acquisition value built on it is arbitrary.
rows = []
for fid in bbo.FUNC_IDS:
    X, y, _ = bbo.load(fid, up_to=PRIOR)
    r2 = bbo.fit(fid, up_to=PRIOR).loo_r2() if len(y) >= 4 else float("nan")
    rows.append(dict(func=f"F{fid}", n_data=len(y), loo_r2=round(r2, 3),
                     verdict="broken" if r2 < 0 else "usable" if r2 == r2 else "too few points"))
pd.DataFrame(rows)


### Anchor audit


In [ ]:
ANCHOR = {
    1: [0.018957, 0.259878],
    2: [0.098559, 0.954719],
    3: [0.159998, 0.011915, 0.958587],
    4: [0.611147, 0.607958, 0.671974, 0.601141],
    5: [0.014852, 0.297741, 0.718557, 0.219953],
    6: [0.398747, 0.385554, 0.570014, 0.711777, 0.389141],
    7: [0.151858, 0.148558, 0.071547, 0.258484, 0.285157, 0.741141],
    8: [0.159174, 0.118198, 0.137956, 0.716535, 0.781515, 0.543548, 0.279585, 0.258543],
}
# Anchor audit: is each proposal being generated from the best point on record?
# This is the check whose absence cost the campaign most of its final score.
for fid in bbo.FUNC_IDS:
    w = bbo.anchor_check(fid, np.array(ANCHOR[fid], float), up_to=PRIOR)
    print(f"F{fid}: {w if w else 'anchored on best-known point'}")


## 3. Visualise

Best-so-far trajectory per function, truncated to the data available this round.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for ax, fid in zip(axes.ravel(), bbo.FUNC_IDS):
    try:
        _, y, rounds = bbo.load(fid, up_to=PRIOR)
    except ValueError:
        ax.set_title(f"F{fid}: no data"); continue
    ax.plot(rounds, y, "o", ms=4, alpha=.55)
    ax.plot(rounds, np.maximum.accumulate(y), "-", lw=2)
    ax.set_title(f"F{fid} (d={bbo.DIMS[fid]})", fontsize=9)
    ax.tick_params(labelsize=7); ax.set_xlabel("round", fontsize=8)
fig.suptitle(f"Best so far through round {PRIOR}", fontsize=11)
fig.tight_layout(); fig.savefig(f"{OUTDIR}/trajectories.png", dpi=140)
plt.show()


## 4. Submission strings


In [ ]:
# Portal format: six decimals, dash-separated, one line per function, no labels.
for fid in bbo.FUNC_IDS:
    print(bbo.submission(proposals[fid]))


## 5. After the portal returns each y

Returns recorded below and folded into `bbo.HISTORY` so the next round sees them.


In [ ]:
# Week 3 portal returns - already folded into bbo.HISTORY.
# returned_y = {
#     1: ...,
#     2: ...,
#     3: ...,
#     4: ...,
#     5: ...,
#     6: ...,
#     7: ...,
#     8: ...,
# }
#
# Returns for this round are not in the working record. The perturbation distances
# printed above are the useful artefact: five functions moved less than 0.01, which
# on these surfaces is inside the noise of a single evaluation.
#
# for fid, y in returned_y.items():
#     bbo.append_result(fid, proposals[fid], y, rnd=WEEK)
